# Modeling — **IndoBERT fine-tuning** (Colab/GPU) — *driver*

Notebook ini **tipis**: menyiapkan lingkungan lalu memanggil skrip sumber-kebenaran
**`src/modeling/train_indobert.py`** (logika, hyperparameter, split, evaluasi — semua
di sana, masuk git). Hasil notebook == `python -m src.modeling.train_indobert`.

Model: `indobenchmark/indobert-base-p1` (3 kelas), pembanding deep-learning terhadap
baseline SVM. Membaca SELURUH koleksi **`processed_bert`** (satu dataset full 14k) dari
MongoDB Atlas.

**Sebelum mulai:** Runtime → Change runtime type → **GPU (T4)**. CPU sangat lambat.


## 1. Clone repo (privat) + dependensi

Repo privat di GitHub → clone butuh **Personal Access Token** (scope `repo`).


In [ ]:
import os
from getpass import getpass
GH_TOKEN = os.environ.get('GH_TOKEN') or getpass('GitHub PAT (repo privat, scope repo): ')
![ -d jokowi_sentiment_project ] || git clone https://{GH_TOKEN}@github.com/Arachnoida/jokowi_sentiment_project.git
%cd jokowi_sentiment_project
%pip install -q 'transformers>=4.40' torch 'pymongo[srv]' dnspython certifi scikit-learn matplotlib pandas numpy python-dotenv


## 2. Set MONGO_URI + jalankan training

Tempel `MONGO_URI` (butuh IP allowlist `0.0.0.0/0` agar Colab bisa konek). Skrip melakukan
split kanonik (urut `comment_id`, seed 42) **identik** SVM → test set sama → perbandingan
adil. Hyperparameter default = konfigurasi terbaik (4 epoch, LR 2e-5).


In [ ]:
import os
from getpass import getpass
os.environ['MONGO_URI'] = os.environ.get('MONGO_URI') or getpass('MONGO_URI: ')
# Tambah --model-out indobert_model bila ingin menyimpan bobot model.
!python -m src.modeling.train_indobert


## 3. Lihat hasil + unduh artefak

Skrip menyimpan `indobert_metrics.json` + `indobert_test_confusion.png` di `outputs/reports/`.
**Penting:** unduh JSON-nya dan commit ke `outputs/reports/` repo lokal agar skrip SVM
(`train_svm_full14k`) bisa memuatnya untuk tabel perbandingan akhir.


In [ ]:
import json
from IPython.display import Image, display
m = json.load(open('outputs/reports/indobert_metrics.json'))
print(json.dumps(m, ensure_ascii=False, indent=2))
display(Image('outputs/reports/indobert_test_confusion.png'))


In [ ]:
# (Opsional) unduh artefak ke lokal untuk di-commit ke outputs/reports/
from google.colab import files
files.download('outputs/reports/indobert_metrics.json')
files.download('outputs/reports/indobert_test_confusion.png')


**Bandingkan dengan SVM.** Setelah `indobert_metrics.json` ada di `outputs/reports/`,
jalankan `python -m src.modeling.train_svm_full14k` untuk memunculkan tabel + grafik
perbandingan akhir SVM vs IndoBERT.
